In [ ]:
import numpy as np

def galois_product_unit_node(p: int) -> complex:
    r"""
    Proposition 3.1 (1): 単位境界ノードへの還元
    計算式: N(\zeta_p) = \prod_{k=1}^{p-1} \zeta_p^k
    理論値: 1.0 (奇素数 p の場合)
    """
    if p % 2 == 0:
        raise ValueError("偶数素数(p=2)を除外した奇素数を入力してください。")

    k_vals = np.arange(1, p)
    zeta_k = np.exp(2j * np.pi * k_vals / p)  # \zeta_p^k の全生成
    return np.prod(zeta_k)                   # ガロア積 (代数体ノーム)

def galois_product_prime_node(p: int) -> complex:
    r"""
    Proposition 3.1 (2): 素数格子ノードへの還元
    計算式: N(1 - \zeta_p) = \prod_{k=1}^{p-1} (1 - \zeta_p^k)
    理論値: p
    """
    k_vals = np.arange(1, p)
    zeta_k = np.exp(2j * np.pi * k_vals / p)
    elements = 1.0 - zeta_k                  # 位相差要素 (1 - \zeta_p^k)
    return np.prod(elements)                 # ガロア積 (代数体ノーム)


# --- 奇素数リストによる実証実行 ---
primes = [3, 5, 7, 11, 13, 17]

print("=" * 70)
print(" PPL Phase Reduction Verification (Proposition 3.1)")
print("=" * 70)
print(f"{'Prime (p)':<10} | {'N(zeta_p) (Target: 1.0)':<26} | {'N(1-zeta_p) (Target: p)':<26}")
print("-" * 70)

for p in primes:
    n_unit = galois_product_unit_node(p)
    n_prime = galois_product_prime_node(p)

    # 浮動小数点誤差の範囲内で虚数部が0であることを検証
    assert abs(n_unit.imag) < 1e-10, f"Error: Imaginary component remains in N(zeta_{p})"
    assert abs(n_prime.imag) < 1e-10, f"Error: Imaginary component remains in N(1-zeta_{p})"

    print(f"{p:<10} | {n_unit.real:<26.8f} | {n_prime.real:<26.8f}")

print("-" * 70)
print("検証完了: すべての複素位相の回転運動が、実スカラーノード (1, p) へ還元されました。")

 PPL Phase Reduction Verification (Proposition 3.1)
Prime (p)  | N(zeta_p) (Target: 1.0)    | N(1-zeta_p) (Target: p)   
----------------------------------------------------------------------
3          | 1.00000000                 | 3.00000000                
5          | 1.00000000                 | 5.00000000                
7          | 1.00000000                 | 7.00000000                
11         | 1.00000000                 | 11.00000000               
13         | 1.00000000                 | 13.00000000               
17         | 1.00000000                 | 17.00000000               
----------------------------------------------------------------------
検証完了: すべての複素位相の回転運動が、実スカラーノード (1, p) へ還元されました。


In [ ]:
from typing import List, Set, Union

def count_ppl_complex_surfaces(M: int, P: Union[List[int], Set[int]]) -> int:
    r"""
    PPL空間における独立複素曲面の総数 N_surface を計算する。

    計算式: N_surface = M * \sum_{p \in \mathcal{P}} (p - 1)

    Parameters:
        M (int): 共有実数軸（中央軸）の数 (例: 3D構造の場合は M = 3)
        P (list or set): 対象とする素数の集合 \mathcal{P} (例: [2, 3, 5])

    Returns:
        int: 独立な複素曲面の総数 N_surface
    """
    if M <= 0:
        raise ValueError("実数軸の数 M は1以上の整数である必要があります。")

    for p in P:
        if p < 2:
            raise ValueError(f"無効な素数が指定されています: p = {p}")

    # 各素数巡回体 Q(\zeta_p) の代数的位相自由度 (p - 1) の総和を算出
    total_phase_degrees = sum(p - 1 for p in P)

    # 実数軸数 M を掛け合わせて総曲面数を導出
    N_surface = M * total_phase_degrees
    return N_surface


# --- 論文(式5)の検証テスト (M=3, P={2, 3, 5}) ---
if __name__ == "__main__":
    M_val = 3
    P_set = [2, 3, 5]

    result = count_ppl_complex_surfaces(M_val, P_set)

    print("=" * 60)
    print(" PPL Complex Surface Counting Verification (Definition 2.2)")
    print("=" * 60)
    print(f"共有実数軸数 (M)       : {M_val}")
    print(f"素数集合 P             : {P_set}")
    print(f"計算結果 (N_surface)   : {result} チャンネル")
    print("-" * 60)
    print(f"理論値との一致確認      : {'一致 (21)' if result == 21 else '不一致'}")

 PPL Complex Surface Counting Verification (Definition 2.2)
共有実数軸数 (M)       : 3
素数集合 P             : [2, 3, 5]
計算結果 (N_surface)   : 21 チャンネル
------------------------------------------------------------
理論値との一致確認      : 一致 (21)


In [ ]:
import sympy

def total_phase_sync_volume(limit: int = 50):
    r"""
    PPL空間の1次元スキャンにおける位相同期体積 a(n) を計算する。

    計算式: a(n) = (k_n - 1) * \Omega(k_n)

    Parameters:
        limit (int): 探索する整数の上限値

    Returns:
        list of dict: 各合成数 k_n に対する計算結果のリスト
    """
    results = []
    n_count = 1

    for k in range(4, limit + 1):
        if not sympy.isprime(k):  # 合成数 k_n の判定
            # 素因数分解 (重複度を含む)
            prime_factors = sympy.factorint(k)
            omega_k = sum(prime_factors.values())  # \Omega(k_n)

            additive_deg = k - 1                   # (k_n - 1)
            a_n = additive_deg * omega_k           # a(n)

            results.append({
                "n": n_count,
                "k_n": k,
                "k_n_minus_1": additive_deg,
                "omega_k": omega_k,
                "a_n": a_n
            })
            n_count += 1

    return results

# --- スクリプトの実行と検証出力 ---
if __name__ == "__main__":
    data = total_phase_sync_volume(limit=30)

    print("=" * 65)
    print(" 1D Scanning Total Phase-Synchronization Volume (Definition 2.4)")
    print("=" * 65)
    # \\Omega(k_n) に変更して SyntaxWarning を防止
    print(f"{'n':<4} | {'k_n':<6} | {'(k_n - 1)':<12} | {'\\Omega(k_n)':<12} | {'a(n)':<8}")
    print("-" * 65)

    for row in data:
        print(f"{row['n']:<4} | {row['k_n']:<6} | {row['k_n_minus_1']:<12} | {row['omega_k']:<12} | {row['a_n']:<8}")

    print("-" * 65)
    print("検証完了: 1次元スキャンによる位相同期体積 a(n) の数列が正常に生成されました。")

 1D Scanning Total Phase-Synchronization Volume (Definition 2.4)
n    | k_n    | (k_n - 1)    | \Omega(k_n)  | a(n)    
-----------------------------------------------------------------
1    | 4      | 3            | 2            | 6       
2    | 6      | 5            | 2            | 10      
3    | 8      | 7            | 3            | 21      
4    | 9      | 8            | 2            | 16      
5    | 10     | 9            | 2            | 18      
6    | 12     | 11           | 3            | 33      
7    | 14     | 13           | 2            | 26      
8    | 15     | 14           | 2            | 28      
9    | 16     | 15           | 4            | 60      
10   | 18     | 17           | 3            | 51      
11   | 20     | 19           | 3            | 57      
12   | 21     | 20           | 2            | 40      
13   | 22     | 21           | 2            | 42      
14   | 24     | 23           | 4            | 92      
15   | 25     | 24           | 2            